### Librerias

In [2]:
import pandas as pd

### Carga de datos

In [3]:
df = pd.read_csv('../../data/raw_data 09-03-2026.csv',index_col=0)


Seleccionamos solo las columnas que necesitamos


In [4]:
df=df[['apartment_id','room_type','price','has_availability','availability_30','availability_60','availability_90','availability_365','review_scores_rating','reviews_per_month','city','insert_date']].copy()

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7001 entries, 0 to 7000
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   apartment_id          7001 non-null   int64  
 1   room_type             7001 non-null   object 
 2   price                 6870 non-null   float64
 3   has_availability      6451 non-null   object 
 4   availability_30       7001 non-null   int64  
 5   availability_60       7001 non-null   int64  
 6   availability_90       7001 non-null   int64  
 7   availability_365      7001 non-null   int64  
 8   review_scores_rating  5674 non-null   float64
 9   reviews_per_month     5747 non-null   float64
 10  city                  7001 non-null   object 
 11  insert_date           7001 non-null   object 
dtypes: float64(3), int64(5), object(4)
memory usage: 711.0+ KB


### Limpieza


Nulos

In [6]:
df.isnull().sum()

apartment_id               0
room_type                  0
price                    131
has_availability         550
availability_30            0
availability_60            0
availability_90            0
availability_365           0
review_scores_rating    1327
reviews_per_month       1254
city                       0
insert_date                0
dtype: int64

Para el análisis de esta semana se ha aplicado la siguiente estrategia:

Eliminación en la variable price: Se han descartado las filas sin precio. Al ser la métrica principal del estudio y presentar un porcentaje de nulos mínimo (1.8%), su eliminación directa es la opción más segura para no introducir sesgos mediante imputaciones artificiales.

Conservación en las reseñas: Los valores nulos de las dos columnas de valoraciones se mantienen intactos. Estas ausencias no son errores, sino información de negocio válida (apartamentos nuevos o sin reservas previas), por lo que conservarlos refleja fielmente la realidad del mercado.

In [7]:
df=df.dropna(subset=['price'])

Duplicados

Se ordena el dataset por identificador y fecha para conservar únicamente el registro más reciente en la tabla principal, archivando las versiones anteriores en un DataFrame secundario a modo de histórico.


In [8]:
df = df.sort_values(by=['apartment_id','insert_date'], ascending=[True, False])


dfanunciosantiguos = df[df.duplicated(subset=['apartment_id'],keep='first')].copy()

df = df.drop_duplicates(subset=['apartment_id'], keep='first')


In [9]:
print(f"Apartamentos únicos listos para análisis (DF principal): {len(df)}")
print(f"Registros antiguos guardados en el histórico: {len(dfanunciosantiguos)}")

Apartamentos únicos listos para análisis (DF principal): 6615
Registros antiguos guardados en el histórico: 255


In [10]:
es_unico = df['apartment_id'].is_unique
print(f"¿Son todos los IDs únicos?: {es_unico}")

¿Son todos los IDs únicos?: True


In [11]:
# Estandarización de texto: formato título para los nombres de las ciudades
df['city'] = df['city'].str.title()

### Transformación 

Para garantizar la integridad del análisis, se ha llevado a cabo un proceso de estandarización estructural del dataset. Esto ha incluido el casting de variables (conversión de las columnas a sus tipos de datos nativos correspondientes, como numéricos, booleanos o fechas) para permitir operaciones matemáticas correctas. Asimismo, se han corregido inconsistencias y anomalías de formato detectadas en varias columnas, asegurando que la información sea coherente y esté optimizada para la fase de modelado.

In [12]:
df["reviews_per_month"]=df["reviews_per_month"].astype('Int64')

df['review_scores_rating']=df['review_scores_rating']/10
df['review_scores_rating']=df['review_scores_rating'].astype('Int64')

# Esta línea convierte lo que no sea 'VERDADERO' a False (Asumiendo que si esta nulo es porque no tiene disponibilidad)
df['has_availability'] = df['has_availability'] == 'VERDADERO'


df['insert_date'] = pd.to_datetime(df['insert_date'], dayfirst=True)

### Creación de variables

Se ha generado una nueva variable booleana para identificar los apartamentos con valoraciones superiores a 80

In [13]:
df['reviews 80+']=df['review_scores_rating'] >= 80 

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6615 entries, 0 to 6999
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   apartment_id          6615 non-null   int64         
 1   room_type             6615 non-null   object        
 2   price                 6615 non-null   float64       
 3   has_availability      6615 non-null   bool          
 4   availability_30       6615 non-null   int64         
 5   availability_60       6615 non-null   int64         
 6   availability_90       6615 non-null   int64         
 7   availability_365      6615 non-null   int64         
 8   review_scores_rating  5380 non-null   Int64         
 9   reviews_per_month     5451 non-null   Int64         
 10  city                  6615 non-null   object        
 11  insert_date           6615 non-null   datetime64[ns]
 12  reviews 80+           5380 non-null   boolean       
dtypes: Int64(2), bool(1), b

In [15]:
df

,apartment_id,room_type,price,has_availability,availability_30,availability_60,availability_90,availability_365,review_scores_rating,reviews_per_month,city,insert_date,reviews 80+
0,11964,Private room,400.0,True,7,20,40,130,97,75,Malaga,2018-07-31,True
1,21853,Private room,170.0,True,0,0,0,162,92,52,Madrid,2020-01-10,True
2,32347,Entire home/apt,990.0,True,26,31,31,270,98,142,Sevilla,2019-07-29,True
3,35379,Private room,400.0,True,9,23,49,300,94,306,Barcelona,2020-01-10,True
4,35801,Private room,900.0,True,0,19,49,312,97,39,Girona,2019-02-19,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6995,27237828,Entire home/apt,1500.0,True,22,47,77,78,100,10,Girona,2018-08-30,True
6996,27241318,Entire home/apt,3130.0,True,26,37,37,243,100,7,Mallorca,2020-04-23,True
6997,27244243,Entire home/apt,990.0,True,24,40,40,40,<NA>,<NA>,Girona,2018-08-30,<NA>
6998,27244794,Entire home/apt,720.0,True,0,0,0,0,100,6,Girona,2019-12-31,True


Se genera una archivo CSV con los datos limpios y otro con los anuncios antiguos

In [ ]:
# Generación del CSV bloqueada (el archivo ya se encuentra en el directorio del proyecto).
#df.to_csv('../../data/data_cleaning/clean_data_09-03-2026.csv', index=False)

#dfanunciosantiguos.to_csv('../../data/data_cleaning/Anuncios antiguos_09-03-2026.csv', index=False)